In [ ]:
%pip install -q otter-grader

In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook()

# In-Class Activity 14: Growing and Breaking Decision Trees

[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tools4ds/DS701-Course-Notes-FA26/blob/main/class_activity_notebooks/14-InClass-Exercise-Decision-Trees/14-InClass-Exercise-Decision-Trees.ipynb)

**DS701 — Session 11 (Tue Oct 13, 2026)**

**Time: ~60 minutes.** Work in groups of 2–3.

- Parts **(a)–(c)** are **autograded** and submitted to Gradescope.
- Part **(d)** is open-ended and graded for **participation**.
- You may use AI assistance, but you must be able to explain and justify every
  part of your solution when asked.

## Setup

We use the **Wisconsin breast cancer** dataset that ships with `scikit-learn`:
569 tumors, 30 continuous features, and a binary label (`1` = benign,
`0` = malignant). Nothing is downloaded, so this runs anywhere.

Run the next two cells as-is — everything below depends on them.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 701

data = load_breast_cancer(as_frame=True)
X = data.data                      # DataFrame, 30 columns
y = data.target.to_numpy()         # 1 = benign, 0 = malignant

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)

print(f"features:      {X.shape[1]}")
print(f"train / test:  {len(X_train)} / {len(X_test)}")
print(f"benign share:  train {y_train.mean():.3f}, test {y_test.mean():.3f}")
X_train.head()

In [ ]:
# Colab: fetch the autograder tests for this activity.
import os, sys, urllib.request

if "google.colab" in sys.modules:
    os.makedirs("tests", exist_ok=True)
    BASE = "https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main/class_activity_notebooks/14-InClass-Exercise-Decision-Trees/tests/"
    for _q in ("q1", "q2", "q3", "q5"):
        urllib.request.urlretrieve(BASE + _q + ".py", "tests/" + _q + ".py")

## Part (a1) — Gini impurity by hand (autograded)

Recall the definition from lecture, for a node $t$ with $c$ classes:

$$\textnormal{Gini}(t) = 1 - \sum_{i=0}^{c-1} p_i(t)^2$$

and the **collective (weighted) impurity** of a binary split into children
$v_L, v_R$:

$$I(\textnormal{children}) = \frac{N(v_L)}{N} I(v_L) + \frac{N(v_R)}{N} I(v_R)$$

Implement both with `numpy`. `gini` takes an array-like of **class counts**
(not probabilities) and must handle an empty node (return `0.0`) and any number
of classes.

In [ ]:
def gini(counts):
    """Gini impurity of a node described by its class counts."""
    ...


def weighted_gini(left_counts, right_counts):
    """Weighted average Gini impurity of the two children of a binary split."""
    ...

In [ ]:
grader.check("q1")

## Part (a2) — Find the best split, then check `scikit-learn`'s work (autograded)

Implement the continuous-attribute sweep from lecture:

1. sort the distinct values of the feature,
2. take candidate thresholds **midway between consecutive distinct values**,
3. score each with `weighted_gini` (left = `x <= tau`),
4. return the threshold with the **lowest** weighted impurity.

Then run it on `worst perimeter` and store the results in `best_thr` and
`best_score`. The check below fits a depth-1 `DecisionTreeClassifier` on the
same single column and confirms that you found *exactly* the same split.

In [ ]:
FEATURE = "worst perimeter"


def best_split_for_feature(x, labels, n_classes=2):
    """Return (best_threshold, best_weighted_gini) for one continuous feature."""
    x = np.asarray(x, dtype=float)
    labels = np.asarray(labels)
    ...


best_thr, best_score = best_split_for_feature(X_train[FEATURE], y_train)

root_gini = gini(np.bincount(y_train, minlength=2))
print(f"root Gini                : {root_gini:.6f}")
print(f"best threshold           : {FEATURE} <= {best_thr}")
print(f"weighted child Gini      : {best_score:.6f}")
print(f"Gini gain                : {root_gini - best_score:.6f}")

In [ ]:
grader.check("q2")

## Part (b) — Watch a tree overfit (autograded)

For each depth in `DEPTHS`, fit a `DecisionTreeClassifier(max_depth=d,
random_state=RANDOM_STATE)` on the **training** set and record training and
test accuracy.

Build a DataFrame `depth_results` with **exactly** the columns
`["max_depth", "train_acc", "test_acc"]`, one row per depth in order, then set:

- `best_depth` — the `max_depth` with the highest test accuracy,
- `best_tree_test_acc` — that test accuracy,
- `full_tree` — a tree fit with `max_depth=None` (grown until pure).

In [ ]:
DEPTHS = [1, 2, 3, 4, 5, 6, 8, 12]

...

depth_results["gap"] = depth_results["train_acc"] - depth_results["test_acc"]
print(depth_results.to_string(index=False))
print(f"\nbest depth by test accuracy: {best_depth} "
      f"(test acc {best_tree_test_acc:.4f})")
print(f"fully grown tree: depth {full_tree.get_depth()}, "
      f"train {full_tree.score(X_train, y_train):.4f}, "
      f"test {full_tree.score(X_test, y_test):.4f}")

In [ ]:
grader.check("q3")

Plot the two accuracy curves. This cell is given — just run it.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(depth_results["max_depth"], depth_results["train_acc"],
        "o-", label="train")
ax.plot(depth_results["max_depth"], depth_results["test_acc"],
        "s--", label="test")
ax.axvline(best_depth, color="grey", lw=1, ls=":")
ax.set_xlabel("max_depth")
ax.set_ylabel("accuracy")
ax.set_title("A single decision tree overfits as it deepens")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

<!-- BEGIN QUESTION -->

### Discuss in your group (written, participation)

The training curve rises to 1.0 and stays there while the test curve peaks and
then *falls*. In two or three sentences: which region of the plot is high bias,
which is high variance, and where would `min_samples_leaf` bite differently
from `max_depth`?

*Your group's answer here.*

<!-- END QUESTION -->

## Part (c) — Random forest and feature importances (autograded)

Fit `RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)` on
the training set and set:

- `rf` — the fitted model,
- `rf_train_acc`, `rf_test_acc` — accuracies,
- `importances` — a `pd.Series` of `rf.feature_importances_` **indexed by
  feature name**, sorted descending,
- `top5` — a list of the 5 most important feature names, most important first.

In [ ]:
...

print(f"random forest  train {rf_train_acc:.4f}  test {rf_test_acc:.4f}")
print(f"best single tree (depth {best_depth}) test {best_tree_test_acc:.4f}")
print("\ntop 5 features:")
print(importances.head(5).to_string())

In [ ]:
grader.check("q5")

Plot the top 10 importances. Given — just run it.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
importances.head(10)[::-1].plot.barh(ax=ax)
ax.set_xlabel("impurity-based importance")
ax.set_title("Random forest: top 10 features")
plt.tight_layout()
plt.show()

<!-- BEGIN QUESTION -->

## Part (d) — Break the feature importances (open-ended, participation)

Impurity-based importances are handed to you for free, and they are easy to
misread. Perturb the feature matrix and find out how fragile they are.

Add **two** columns to a copy of both `X_train` and `X_test`:

1. `"noise"` — pure Gaussian noise, unrelated to the label;
2. `"worst perimeter copy"` — an **exact duplicate** of `worst perimeter`, the
   feature the forest liked most.

Refit the forest on the augmented data (call it `rf_aug`, same settings) and
compare against Part (c).

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)


def add_perturbations(df, generator):
    """Return a copy of df with a noise column and a duplicated strong feature."""
    ...


X_train_aug = add_perturbations(X_train, rng)
X_test_aug = add_perturbations(X_test, rng)

...

print(f"augmented forest test accuracy: {rf_aug.score(X_test_aug, y_test):.4f}"
      f"   (was {rf_test_acc:.4f})")
print(f"\nnoise importance : {imp_aug['noise']:.5f}  "
      f"(rank {list(imp_aug.index).index('noise') + 1} of {len(imp_aug)})")
print(f"worst perimeter  : {imp_aug['worst perimeter']:.5f}  "
      f"(was {importances['worst perimeter']:.5f})")
print(f"...its copy      : {imp_aug['worst perimeter copy']:.5f}")
print(f"the two together : "
      f"{imp_aug['worst perimeter'] + imp_aug['worst perimeter copy']:.5f}")
print("\ntop 8 after perturbation:")
print(imp_aug.head(8).to_string())

Now answer, in the cell below:

- Where does `"noise"` rank, and is its importance exactly zero? Why not?
- What happened to the importance of `worst perimeter` now that an identical
  copy exists? What is the *sum* of the two, compared with the original single
  value? Is the forest's accuracy affected?
- If a stakeholder asked "which of these two measurements matters more?", what
  would you tell them?

**Optional extensions** if your group finishes early: swap
`rf.feature_importances_` for
`sklearn.inspection.permutation_importance` on the **test** set and see whether
it tells the same story; or replace `"noise"` with a **high-cardinality ID
column** (`np.arange(len(X_train))`) and connect it to the customer-ID example
from lecture.

> **Be ready to be cold-called.** Staff will circulate and ask one member of
> your group to explain *why* your numbers came out the way they did — not just
> what they were. Everyone in the group should be able to answer.

*Your group's answer here (include your group members' names).*

<!-- END QUESTION -->

## Wrap-up

- Parts (a)–(c): run all cells top to bottom, confirm every `grader.check(...)`
  cell passes, and submit this notebook to the Gradescope assignment
  **Activity 14**.
- The written parts are graded for participation — make sure your group's names
  are in them.